# Ghi chú học tập: Modal và SpecialistAgent

## Tóm tắt quy trình của notebook
Notebook thiết lập môi trường và token cho Modal, chạy thử hàm cục bộ và từ xa, gọi mô hình Llama để định giá sản phẩm, rồi triển khai dịch vụ định giá trên Modal. Cuối cùng, notebook khởi tạo `SpecialistAgent` để kết hợp tiền xử lý và định giá.

## Ý nghĩa chính của notebook
Bài học minh họa cách đưa một tác vụ AI từ máy cục bộ lên hạ tầng Modal. Dữ liệu đầu vào là mô tả sản phẩm; dữ liệu được tiền xử lý, gửi đến dịch vụ định giá và trả về mức giá dự đoán.

## Mục tiêu cuối cùng
Sau khi hoàn thành, bạn có thể cấu hình Modal, triển khai một hàm hoặc lớp Python lên cloud và gọi `SpecialistAgent` để ước lượng giá sản phẩm.

# Chào mừng đến với Week 8 có nhiều nội dung

## Tuần này chúng ta có nhiều việc cần thực hiện

Tốc độ học sẽ nhanh hơn bình thường vì bạn đang dần thành thạo kỹ năng của một kỹ sư LLM.

# The Price is Right

## Lộ trình Week 8

Ngày 1: Modal.com và SpecialistAgent  
Ngày 2: RAG, FrontierAgent, Ensemble Agent  
Ngày 3: ScannerAgent, MessengerAgent  
Ngày 4: AutonomousPlannerAgent và DealAgentFramework  
Ngày 5: Phần kết The Price Is Right



<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#b22;">Rất quan trọng: cập nhật mã nguồn mới nhất</h2>
            <span style="color:#b22;">Các bài lab luôn được cải thiện, bổ sung thêm ví dụ và bài tập.
            Khi bắt đầu mỗi tuần, hãy kiểm tra xem bạn đã có mã nguồn mới nhất chưa.<br/>
            Xem Guide 3 trong thư mục Guides nếu bạn chưa biết cách chạy <code>git pull</code>.
            </span>
        </td>
    </tr>
</table>

In [36]:
# Nạp thư viện và biến môi trường cần cho các cell tiếp theo.
import os
import locale
import tomllib
from pathlib import Path
import modal
from agents.preprocessor import Preprocessor
from dotenv import load_dotenv
load_dotenv(override=True)  # Ưu tiên giá trị trong file .env.

# Modal cần đồng thời cả hai giá trị để xác thực.
modal_token_id = os.getenv("MODAL_TOKEN_ID", "").strip()
modal_token_secret = os.getenv("MODAL_TOKEN_SECRET", "").strip()
if not modal_token_id or not modal_token_secret:
    # Trên Windows, nạp profile có đủ token nếu Modal không tự thấy file cấu hình.
    modal_config_path = Path.home() / ".modal.toml"
    if modal_config_path.exists():
        with modal_config_path.open("rb") as config_file:
            modal_config = tomllib.load(config_file)
        profiles = [
            value for value in modal_config.values() if isinstance(value, dict)
        ]
        profile_config = next(
            (
                profile
                for profile in profiles
                if profile.get("token_id") and profile.get("token_secret")
            ),
            {},
        )
        modal_token_id = str(profile_config.get("token_id", "")).strip()
        modal_token_secret = str(profile_config.get("token_secret", "")).strip()
        if modal_token_id and modal_token_secret:
            os.environ["MODAL_TOKEN_ID"] = modal_token_id
            os.environ["MODAL_TOKEN_SECRET"] = modal_token_secret

if not modal_token_id or not modal_token_secret:
    raise RuntimeError(
        "Thiếu thông tin xác thực Modal. Hãy chạy `uv run modal token new` "
        "hoặc thêm MODAL_TOKEN_ID và MODAL_TOKEN_SECRET vào file .env, "
        "sau đó khởi động lại kernel và chạy lại cell này."
    )

In [37]:
# Kiểm tra máy có thể hiển thị ký tự đặc biệt bằng mã hóa UTF-8.

print(locale.getpreferredencoding())  # Kết quả mong đợi: 'UTF-8'.

cp1252


In [38]:
# Ép đầu ra của Python dùng UTF-8 để tránh lỗi ký tự trên Windows.
os.environ["PYTHONIOENCODING"] = "utf-8"

# Thiết lập token cho Modal

## Quan trọng: hãy đọc và làm theo các bước sau

Trước tiên, truy cập: https://modal.com

Đăng ký tài khoản, sau đó mở menu Avatar ở góc trên bên phải và chọn "Settings".

Chọn "API Tokens" ở thanh bên trái, rồi chọn "New Token".

Modal sẽ cung cấp lệnh tương tự:

`modal token set --token-id ak-somethinghere --token-secret as-somethinghere`

Vì dự án dùng `uv`, lệnh cần chạy là:

`uv run modal token set --token-id ak-somethinghere --token-secret as-somethinghere`

### Khắc phục sự cố

Nếu gặp lỗi, hãy thử một trong ba cách:

1. Chạy `uv run modal token new` trước `uv run modal token set ...`.  

2. Gợi ý cho người dùng Windows:

> Nếu chạy `modal token new` nhưng vẫn gặp lỗi xác thực, hãy kiểm tra file `.modal.toml`. Trên Windows, file này có thể được tạo trong thư mục hồ sơ người dùng mà môi trường ảo không đọc được. Hãy thử chép file vào thư mục đang chạy bài lab.

3. Cấu hình thủ công:

Bạn có thể thêm trực tiếp hai khóa vào file `.env`:

```
MODAL_TOKEN_ID=ak-...
MODAL_TOKEN_SECRET=as-...
```

Sau đó chạy lại `load_dotenv(override=True)` để nạp các biến môi trường.

In [39]:
# Nạp Modal app và hai hàm minh họa cách chạy tại các khu vực khác nhau.
from hello import app, hello, hello_europe

In [40]:
# Chạy hàm ngay trên máy hiện tại để so sánh với lời gọi từ xa.
with app.run():
    reply=hello.local()
reply

'Hello from Ho Chi Minh City, Ho Chi Minh City (HCMC), VN!!'

In [41]:
# Gửi lời gọi đến hạ tầng Modal; Modal thực thi hàm trong môi trường từ xa.
with app.run():
    reply=hello.remote()
reply

'Hello from Columbus, Ohio, US!!'

## Hàm `hello_europe`

Trong `hello.py` có thêm hàm đơn giản `hello_europe`.

Hàm dùng decorator:  
`@app.function(image=image, region="eu")`

Xem kết quả ở cell dưới. Tham khảo thêm cấu hình khu vực tại [tài liệu Modal](https://modal.com/docs/guide/region-selection).

Lưu ý: chỉ định khu vực có thể tiêu tốn thêm một ít credit.

In [43]:
# Gọi hàm được cấu hình chạy tại khu vực châu Âu.
try:
    with app.run():
        reply = hello_europe.remote()
    reply
except Exception as error:
    if "App create rate limit exceeded" in str(error):
        raise RuntimeError(
            "Modal đang giới hạn số lần tạo app tạm thời. "
            "Hãy đợi vài phút, không chạy lặp cell này, rồi thử lại một lần."
        ) from error
    raise

# Trước khi tiếp tục

## Cần lưu Hugging Face Token thành secret trong Modal

## Đây là bước rất quan trọng

Secret trong Modal có một **tên** để mã nguồn tham chiếu.  
Bên trong secret là một KEY và VALUE.  
Ta sẽ tạo secret với:  

Name: huggingface-secret  
Key: HF_TOKEN  
Value: hf_...  

## Các bước thực hiện:

1. Đăng nhập modal.com và mở dashboard.  
2. Chọn Secrets trên thanh điều hướng.  
3. Tạo secret mới, chọn Hugging Face và đặt tên **huggingface-secret** vì mã nguồn sẽ dùng tên này.  
4. Điền key là `HF_TOKEN` và value là token thật bắt đầu bằng `hf_...`.  
5. Chọn Done.

### Bây giờ chúng ta làm việc với Llama

In [44]:
# Cảnh báo deprecation khi thêm module cục bộ vào Image có thể được bỏ qua.
# Cell này nạp app Llama và hàm sinh văn bản từ xa.

from llama import app, generate

In [45]:
# Hiển thị log Modal và gọi mô hình Llama để hoàn thành đoạn văn bản.
with modal.enable_output():
    with app.run():
        result=generate.remote("Never gonna give you up, never gonna")
result

# Có thể thay prompt bằng: "Hey Jude, don't make it".

✓ Initialized. View run at 
https://modal.com/apps/cuongphan96-tech/main/ap-ObngaYrA5ZySrCxzug2Aq1
- Initializing...
\ Creating objects...objects...
└── - Creating mount 
    c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\llama.py: Uploaded 
/ Creating objects...
└── | Creating mount 
    c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\llama.py: Uploaded 
\ Creating objects...
└── - Creating mount 
    c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\llama.py: Finalizing 
- Creating objects...
└── | Creating mount 
    c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\llama.py: Finalizing 
\ Creating objects...
├── 🔨 Created mount 
│   c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\llama.py
/ Creating objects...
├── 🔨 Created mount 
│   c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\llama.py
\ Creating objects...
├── 🔨 Created mount 
│   c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\llama.py
└── 🔨 Created function generate.
✓ Created objects.

'<|begin_of_text|>Never gonna give you up, never gonna let you down, never'

In [48]:
# Nạp app tạm thời và hàm định giá sản phẩm trên Modal.
from pricer_ephemeral import app, price

## Lưu ý về GPU và thanh toán Modal

Hàm `price` trong `pricer_ephemeral.py` dùng GPU `T4` và CUDA để chạy mô hình Llama. Modal yêu cầu tài khoản có phương thức thanh toán trước khi tạo hàm GPU, vì vậy token đúng vẫn có thể gặp lỗi: `Please add a payment method to use T4 GPU functions.`

Để chạy cell định giá, hãy mở phần Billing/Settings trong Modal và thêm phương thức thanh toán. Việc thêm phương thức thanh toán không đồng nghĩa với việc bị tính phí ngay; chi phí chỉ phát sinh khi tài nguyên được sử dụng theo chính sách của Modal.

Sau khi cập nhật tài khoản, hãy khởi động lại cell gọi `price`.

In [49]:
# Gửi mô tả micro đến hàm định giá từ xa và xem kết quả.
try:
    with modal.enable_output():
        with app.run():
            result = price.remote("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
    result
except Exception as error:
    if "Please add a payment method" in str(error):
        raise RuntimeError(
            "Modal đã xác thực thành công nhưng tài khoản chưa có phương thức thanh toán. "
            "Hàm price cần GPU T4; hãy thêm phương thức thanh toán trong Modal rồi chạy lại cell này."
        ) from error
    raise

✓ Initialized. View run at 
https://modal.com/apps/cuongphan96-tech/main/ap-FfVyalGiD1TPLVN5q5Gsgi
- Initializing...
| Creating objects...objects...
└── - Creating mount 
    c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\pricer_ephemeral.py: 
- Creating objects...
└── | Creating mount 
    c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\pricer_ephemeral.py: 
| Creating objects...
└── / Creating mount 
    c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\pricer_ephemeral.py: 
- Creating objects...
└── | Creating mount 
    c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\pricer_ephemeral.py: 
| Creating objects...
└── - Creating mount 
    c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\pricer_ephemeral.py: 
- Creating objects...
└── | Creating mount 
    c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\pricer_ephemeral.py: 
| Creating objects...
└── - Creating mount 
    c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\pricer_ephemeral.py: 
- Cre

In [50]:
# Tiền xử lý mô tả sản phẩm để tạo đầu vào chuẩn hơn cho bộ định giá.
preprocessor = Preprocessor()
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
print(text)

### System:
Title: HyperX Quadcast Gaming Microphone
Category: Electronics
Brand: HyperX
Description: High-quality condenser microphone with a wide pickup range for clear and detailed sound.
Details: Features a durable metal grill, built-in pop filter, and USB-C connectivity for easy plug-and-play installation.


In [51]:
# Dùng mô hình Groq cụ thể thay cho mô hình tiền xử lý mặc định.
preprocessor = Preprocessor(model_name="groq/openai/gpt-oss-20b")
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
print(text)

Title: HyperX Quadcast USB‑C Mic  
Category: Audio Equipment  
Brand: HyperX  
Description: The HyperX Quadcast USB‑C mic delivers crystal‑clear audio for streaming, gaming, and podcasting.  
Details: With a built‑in condenser capsule, four‑way directional pattern selection, and plug‑and‑play USB‑C connectivity, setup is effortless.


### Thêm biến này vào `.env` nếu muốn Preprocessor dùng mô hình khác theo mặc định:

`PRICER_PREPROCESSOR_MODEL=groq/openai/gpt-oss-20b`

In [52]:
# Định giá mô tả đã được tiền xử lý để so sánh với đầu vào thô.
with modal.enable_output():
    with app.run():
        result = price.remote(text)
print(result)

✓ Initialized. View run at 
https://modal.com/apps/cuongphan96-tech/main/ap-jUy88qiM8Se5d3tGlPhZHB
- Initializing...
/ Creating objects...objects...
└── - Creating mount 
    c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\pricer_ephemeral.py: 
| Creating objects...
└── | Creating mount 
    c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\pricer_ephemeral.py: 
- Creating objects...
└── - Creating mount 
    c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\pricer_ephemeral.py: 
| Creating objects...
└── | Creating mount 
    c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\pricer_ephemeral.py: 
- Creating objects...
├── 🔨 Created mount 
│   c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\pricer_ephemeral.py
| Creating objects...
├── 🔨 Created mount 
│   c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\pricer_ephemeral.py
- Creating objects...
├── 🔨 Created mount 
│   c:\Users\user\Desktop\llm_engineering_CuongPhan\week8\pricer_ephemeral.py
| Creating obj

In [53]:
# Kiểm tra lại văn bản chuẩn hóa đã được gửi sang dịch vụ định giá.
print(text)

Title: HyperX Quadcast USB‑C Mic  
Category: Audio Equipment  
Brand: HyperX  
Description: The HyperX Quadcast USB‑C mic delivers crystal‑clear audio for streaming, gaming, and podcasting.  
Details: With a built‑in condenser capsule, four‑way directional pattern selection, and plug‑and‑play USB‑C connectivity, setup is effortless.


## Chuyển từ ứng dụng tạm thời sang ứng dụng đã triển khai

Từ dòng lệnh, `uv run modal deploy xxx` sẽ triển khai mã nguồn thành Deployed App.

Đây là cách đóng gói dịch vụ AI sau API để dùng trong hệ thống production.

Bạn cũng có thể tạo REST endpoint, nhưng notebook này gọi trực tiếp từ Python.

## Lưu ý về secrets

Trong `pricer_service.py` và `pricer_service2.py` có đoạn mã gần đầu file:  
`secrets = [modal.Secret.from_name("hf-secret")]`  
Bạn có thể cần đổi `hf-secret` thành `huggingface-secret` tùy theo tên secret đã tạo trên Modal.  
Để kiểm tra, truy cập trang sau và xem cột đầu tiên:  
https://modal.com/secrets/

## Lưu ý cho người dùng Windows

Cell kế tiếp chạy `uv run modal deploy` trong Jupyter. Một số phiên bản Windows có thể gặp lỗi Unicode do Modal in ký tự không hiển thị được. Khi đó, hãy mở Terminal và chạy lệnh triển khai tại đó.

In [ ]:
# Triển khai dịch vụ định giá từ thư mục gốc của repository.
# Dùng đường dẫn file để Modal tìm được module trong thư mục week8.
!uv run modal deploy week8/pricer_service.py

In [54]:
# Lấy tham chiếu đến hàm đã triển khai bằng tên app và tên hàm.
pricer = modal.Function.from_name("pricer-service", "price")

Theo dõi quá trình triển khai tại:

https://modal.com

In [55]:
# Lời gọi đầu tiên có thể mất thời gian vì container cần khởi động.

pricer.remote(text)

90.0

In [ ]:
# Triển khai phiên bản dịch vụ dùng class từ thư mục gốc của repository.
!uv run modal deploy week8/pricer_service2.py

In [59]:
# Lấy class đã triển khai trong app class, tạo instance và gọi phương thức định giá từ xa.
Pricer = modal.Cls.from_name("pricer-service-class", "Pricer")
pricer = Pricer()
reply = pricer.price.remote(text)
print(reply)

90.0


In [60]:
# Gọi lại cùng instance để quan sát việc tái sử dụng dịch vụ đang hoạt động.
reply = pricer.price.remote(text)
print(reply)

90.0


# Tùy chọn: giữ Modal ở trạng thái sẵn sàng

## Cách cải thiện tốc độ dịch vụ định giá Modal

Lần đầu chạy class Modal có thể mất đến 10 phút để build.  
Những lần sau sẽ nhanh hơn: khoảng 30 giây nếu container phải khởi động lại, hoặc 2 giây khi còn sẵn sàng.  
Để luôn gần 2 giây, có thể ngăn container ngủ bằng cách chỉnh hằng số trong `pricer_service2.py`:

`MIN_CONTAINERS = 0`



Đặt thành 1 để giữ một container hoạt động.  
Lưu ý: cách này tốn credit vì tiến trình chạy liên tục.

Hoặc chạy đoạn mã dưới để dịch vụ giữ ấm 20 phút thay vì 2 phút.

### Mã giữ ấm 20 phút trước khi hạ nhiệt:

```python
import modal
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()
pricer.update_autoscaler(scaledown_window=1200)
```

### Mã đưa thời gian giữ ấm về 2 phút

```python
import modal
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()
pricer.update_autoscaler(scaledown_window=120)
```

## Giới thiệu lớp Agent

Theo mặc định, Agent tiền xử lý dữ liệu bằng Llama3.2.

Nếu muốn dùng Groq, thêm biến môi trường sau:

```
PRICER_PREPROCESSOR_MODEL=groq/openai/gpt-oss-20b
```

In [73]:
# Bật mức log INFO để theo dõi quá trình làm việc của Agent.
import logging
root = logging.getLogger()
root.setLevel(logging.INFO)

In [74]:
# Nạp lại SpecialistAgent để notebook dùng tên app class Modal mới.
import importlib
import sys
sys.modules.pop("agents.specialist_agent", None)
from agents.specialist_agent import SpecialistAgent

In [75]:
# Khởi tạo Agent để tái sử dụng cho các yêu cầu định giá.
agent = SpecialistAgent()


INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal


In [81]:
# Yêu cầu Agent ước lượng giá cho sản phẩm; đây là kết quả cuối của quy trình.
agent.price("iPhone 10")

INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $350.00


350.0

In [ ]:
# Cell trống để bạn thử thêm các yêu cầu định giá khác với Agent.